# Full-data CatBoost benchmark with semantic features

Цель ноутбука - честно проверить вклад semantic features поверх уже существующего сильного CatBoost baseline.

Сравниваются две модели на полном temporal split:

```text
1. safe_catboost_plus_popularity
2. safe_catboost_plus_popularity + semantic features
```

Протокол специально повторяет логику исходного baseline:
- тот же список safe-признаков;
- та же train-only popularity логика;
- OOF popularity encoding для train;
- popularity для val/test считается только по train;
- threshold выбирается только на validation;
- leaky/risky признаки не используются.

Важно: это не "одинаковый размер train с QLoRA", а основной full-data CatBoost reference.  
QLoRA можно сравнивать отдельно как LLM-ветку, обученную на 20k examples.


In [ ]:
# Optional dependency installation:
# !pip install -q catboost pandas pyarrow scikit-learn


In [ ]:
import os, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("F:/Coursework")

print("PROJECT_ROOT:", PROJECT_ROOT)


## 1. Configuration


In [ ]:
LABEL_COL = "label_strict"
ITEM_COL = "target_app_id"
METHOD_BASE = "safe_catboost_plus_popularity_reproduced"
METHOD_SEM = "safe_catboost_plus_popularity_semantic"
ALPHA = 10.0
USE_GPU = False

DATA_DIR = PROJECT_ROOT / "data" / "final" / "tabular_temporal"
SEMANTIC_DIR = PROJECT_ROOT / "outputs" / "semantic"
SEMANTIC_ZIP = SEMANTIC_DIR / "steam_semantic_outputs.zip"
OUT_DIR = PROJECT_ROOT / "outputs" / "semantic" / "full_catboost_semantic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)


## 2. Load tabular data


In [ ]:
train_raw = pd.read_parquet(DATA_DIR / "train_tabular.parquet")
val_raw = pd.read_parquet(DATA_DIR / "val_tabular.parquet")
test_raw = pd.read_parquet(DATA_DIR / "test_tabular.parquet")

print("shapes:", train_raw.shape, val_raw.shape, test_raw.shape)
print("label balance train:", train_raw[LABEL_COL].value_counts().to_dict())
print("label balance val:", val_raw[LABEL_COL].value_counts().to_dict())
print("label balance test:", test_raw[LABEL_COL].value_counts().to_dict())
print("columns:", train_raw.columns.tolist())


## 3. Exact baseline feature list


In [ ]:
BASE_FEATURES = [
    "history_len",
    "history_positive_count",
    "history_negative_count",
    "history_positive_share",
    "history_total_hours",
    "history_mean_hours",
    "history_median_hours",
    "history_max_hours",
    "history_min_hours",
    "history_liked_mean_hours",
    "history_disliked_mean_hours",
    "target_token_count",
    "liked_token_count",
    "disliked_token_count",
    "target_liked_overlap_count",
    "target_disliked_overlap_count",
    "target_liked_jaccard",
    "target_disliked_jaccard",
    "target_jaccard_diff",
    "target_description_len",
    "target_title_len",
    "price",
    "required_age",
    "dlc_count",
    "release_year",
]

POP_FEATURES = [
    "train_item_popularity_score",
    "train_item_log_count",
]

FEATURES_BASE = BASE_FEATURES + POP_FEATURES

missing = [c for c in BASE_FEATURES if c not in train_raw.columns]
if missing:
    raise ValueError(f"Required baseline features are missing: {missing}")

print("base features:", len(BASE_FEATURES))
print("popularity features:", POP_FEATURES)


## 4. Train-only popularity logic


In [ ]:
def build_item_stats(df, global_rate):
    stats = df.groupby(ITEM_COL)[LABEL_COL].agg(["sum", "count"]).reset_index()
    stats["train_item_popularity_score"] = (stats["sum"] + ALPHA * global_rate) / (stats["count"] + ALPHA)
    stats["train_item_log_count"] = np.log1p(stats["count"])
    return stats[[ITEM_COL, "train_item_popularity_score", "train_item_log_count"]]

def add_full_train_popularity(df, train):
    global_rate = float(train[LABEL_COL].mean())
    stats = build_item_stats(train, global_rate)
    result = df.merge(stats, on=ITEM_COL, how="left")
    result["train_item_popularity_score"] = result["train_item_popularity_score"].fillna(global_rate)
    result["train_item_log_count"] = result["train_item_log_count"].fillna(0.0)
    return result

def add_oof_train_popularity(train):
    result = train.copy()
    result["train_item_popularity_score"] = np.nan
    result["train_item_log_count"] = np.nan
    y = train[LABEL_COL].astype(int).values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fit_idx, hold_idx in skf.split(train, y):
        fit_part = train.iloc[fit_idx]
        hold_part = train.iloc[hold_idx]
        global_rate = float(fit_part[LABEL_COL].mean())
        stats = build_item_stats(fit_part, global_rate)
        encoded = hold_part[[ITEM_COL]].merge(stats, on=ITEM_COL, how="left")
        result.iloc[hold_idx, result.columns.get_loc("train_item_popularity_score")] = encoded["train_item_popularity_score"].fillna(global_rate).values
        result.iloc[hold_idx, result.columns.get_loc("train_item_log_count")] = encoded["train_item_log_count"].fillna(0.0).values
    return result

train = add_oof_train_popularity(train_raw)
val = add_full_train_popularity(val_raw, train_raw)
test = add_full_train_popularity(test_raw, train_raw)

print("Popularity feature means:")
print("train popularity mean:", train["train_item_popularity_score"].mean())
print("val popularity mean:", val["train_item_popularity_score"].mean())
print("test popularity mean:", test["train_item_popularity_score"].mean())


## 5. Load and merge semantic features


In [ ]:
SEM_COLS = [
    "semantic_target_liked_cosine",
    "semantic_target_disliked_cosine",
    "semantic_similarity_gap",
    "semantic_max_liked_cosine",
    "semantic_max_disliked_cosine",
    "semantic_top3_liked_mean_cosine",
    "semantic_top3_disliked_mean_cosine",
]

expected_semantic_files = [
    "train_semantic_features.parquet",
    "val_semantic_features.parquet",
    "test_semantic_features.parquet",
]

if SEMANTIC_ZIP.exists():
    missing = [name for name in expected_semantic_files if not list(SEMANTIC_DIR.rglob(name))]
    if missing:
        with zipfile.ZipFile(SEMANTIC_ZIP, "r") as z:
            z.extractall(SEMANTIC_DIR)
        print("Semantic zip extracted.")
    else:
        print("Semantic files already extracted.")

def find_semantic_file(name):
    hits = list(SEMANTIC_DIR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Required semantic file was not found: {name}")
    return hits[0]

train_sem = pd.read_parquet(find_semantic_file("train_semantic_features.parquet"))
val_sem = pd.read_parquet(find_semantic_file("val_semantic_features.parquet"))
test_sem = pd.read_parquet(find_semantic_file("test_semantic_features.parquet"))

print("semantic shapes:", train_sem.shape, val_sem.shape, test_sem.shape)
print("semantic columns:", train_sem.columns.tolist())


In [ ]:
def merge_semantic(tab, sem):
    sem_cols = [c for c in SEM_COLS if c in sem.columns]
    if "sample_id" in tab.columns and "sample_id" in sem.columns:
        return tab.merge(sem[["sample_id"] + sem_cols], on="sample_id", how="left")
    if len(tab) != len(sem):
        raise ValueError("Semantic and tabular datasets have different number of rows; row-wise merge is not safe.")
    return pd.concat([tab.reset_index(drop=True), sem[sem_cols].reset_index(drop=True)], axis=1)

train_semantic = merge_semantic(train, train_sem)
val_semantic = merge_semantic(val, val_sem)
test_semantic = merge_semantic(test, test_sem)

semantic_features = [c for c in SEM_COLS if c in train_semantic.columns]
FEATURES_SEM = FEATURES_BASE + semantic_features

print("semantic features:", semantic_features)
print("features base:", len(FEATURES_BASE))
print("features semantic:", len(FEATURES_SEM))
print("semantic NaN share:")
print(train_semantic[semantic_features].isna().mean())


## 6. Feature safety validation


In [ ]:
FORBIDDEN_EXACT = {
    "label_strict",
    "label_hybrid",
    "label_hours_relative",
    "is_recommended",
    "output",
    "target_hours",
    "target_hours_aux",
    "user_id",
    "target_app_id",
    "app_id",
    "review_id",
    "positive_ratio",
    "user_reviews",
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "average_playtime_forever",
    "median_playtime_forever",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent",
}

FORBIDDEN_PATTERNS = [
    "label",
    "output",
    "target_hours",
    "review_id",
    "hours_relative",
    "hours_reference",
    "low_hours_positive",
    "high_hours_negative",
]

def validate_features(df, features, name):
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"{name} has missing columns: {missing}")
    bad = []
    for c in features:
        cl = c.lower()
        if c in FORBIDDEN_EXACT or any(p in cl for p in FORBIDDEN_PATTERNS):
            bad.append(c)
        elif not pd.api.types.is_numeric_dtype(df[c]):
            bad.append(c)
    if bad:
        raise ValueError(f"{name} contains forbidden or non-numeric features: {bad}")

validate_features(train, FEATURES_BASE, "FEATURES_BASE")
validate_features(train_semantic, FEATURES_SEM, "FEATURES_SEM")

print("Feature validation passed.")


## 7. Training and metrics


In [ ]:
def calc_metrics(y_true, score, threshold):
    pred = score >= threshold
    return {
        "roc_auc": float(roc_auc_score(y_true, score)),
        "pr_auc": float(average_precision_score(y_true, score)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold),
    }

def choose_threshold(y_true, score):
    thresholds = np.linspace(0.0, 1.0, 1001)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        pred = score >= threshold
        cur_f1 = f1_score(y_true, pred, zero_division=0)
        if cur_f1 > best_f1:
            best_f1 = cur_f1
            best_threshold = threshold
    return float(best_threshold)

def train_eval_catboost(method, train_df, val_df, test_df, features):
    X_train = train_df[features]
    X_val = val_df[features]
    X_test = test_df[features]

    y_train = train_df[LABEL_COL].astype(int).values
    y_val = val_df[LABEL_COL].astype(int).values
    y_test = test_df[LABEL_COL].astype(int).values

    params = {
        "iterations": 1000,
        "learning_rate": 0.04,
        "depth": 6,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": 100,
        "allow_writing_files": False,
        "early_stopping_rounds": 80,
    }
    if USE_GPU:
        params["task_type"] = "GPU"
        params["devices"] = "0"

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test)[:, 1]

    threshold = choose_threshold(y_val, val_score)
    val_metrics = calc_metrics(y_val, val_score, threshold)
    test_metrics = calc_metrics(y_test, test_score, threshold)

    feature_importance = pd.DataFrame({
        "feature": features,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    model.save_model(OUT_DIR / f"{method}.cbm")
    feature_importance.to_csv(OUT_DIR / f"{method}_feature_importance.csv", index=False)
    pd.DataFrame({"y_true": y_val, "score": val_score, "pred": (val_score >= threshold).astype(int)}).to_csv(OUT_DIR / f"{method}_val_predictions.csv", index=False)
    pd.DataFrame({"y_true": y_test, "score": test_score, "pred": (test_score >= threshold).astype(int)}).to_csv(OUT_DIR / f"{method}_test_predictions.csv", index=False)

    return {
        "method": method,
        "val_roc_auc": val_metrics["roc_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "threshold": threshold,
        "best_iteration": model.get_best_iteration(),
        "n_features": len(features),
    }

rows = []
rows.append(train_eval_catboost(METHOD_BASE, train, val, test, FEATURES_BASE))
rows.append(train_eval_catboost(METHOD_SEM, train_semantic, val_semantic, test_semantic, FEATURES_SEM))

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / "metrics_full_catboost_semantic.csv", index=False)
metrics_df


## 8. Compare against saved baseline and QLoRA reference


In [ ]:
saved_baseline_path = PROJECT_ROOT / "outputs" / "baselines" / "metrics_baselines.csv"
if saved_baseline_path.exists():
    saved_baselines = pd.read_csv(saved_baseline_path)
    display(saved_baselines[saved_baselines["method"].astype(str).str.contains("safe_catboost_plus_popularity", case=False, na=False)])

QLORA_REFERENCE = {
    "method": "qwen3_4b_qlora_checkpoint2500",
    "val_roc_auc": 0.7962935,
    "test_roc_auc": 0.795959,
    "test_pr_auc": 0.7900838560066099,
    "test_accuracy": 0.7145,
    "test_precision": 0.6771263418662262,
    "test_recall": 0.82,
    "test_f1": 0.741745816372682,
    "threshold": 0.44,
    "best_iteration": None,
    "n_features": None,
}

comparison_df = pd.concat([metrics_df, pd.DataFrame([QLORA_REFERENCE])], ignore_index=True)
comparison_df.to_csv(OUT_DIR / "metrics_full_catboost_semantic_with_qlora_reference.csv", index=False)
display(comparison_df)


## 9. Summary report


In [ ]:
summary = {
    "task": "full_catboost_semantic_benchmark",
    "protocol": {
        "train": "full temporal train split",
        "val": "full temporal validation split",
        "test": "full temporal test split",
        "popularity": "OOF on train, full train encoding for val/test",
        "threshold": "selected on validation only",
    },
    "feature_sets": {
        "base_features": BASE_FEATURES,
        "popularity_features": POP_FEATURES,
        "semantic_features": semantic_features,
        "features_base": FEATURES_BASE,
        "features_semantic": FEATURES_SEM,
    },
    "metrics": comparison_df.to_dict(orient="records"),
}

with open(OUT_DIR / "full_catboost_semantic_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

report_path = PROJECT_ROOT / "reports" / "experiment_reports" / "full_catboost_semantic_summary.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

report_text = "# Full-data CatBoost semantic benchmark\n\n"
report_text += "This report compares the reproduced safe CatBoost + train-only popularity baseline with the same model extended by semantic embedding features.\n\n"
report_text += "The protocol follows the original train-only popularity logic: OOF encoding for train and full-train encoding for validation/test.\n\n"
report_text += comparison_df.to_string(index=False)
report_text += "\n"

report_path.write_text(report_text, encoding="utf-8")

print("saved metrics:", OUT_DIR / "metrics_full_catboost_semantic.csv")
print("saved comparison:", OUT_DIR / "metrics_full_catboost_semantic_with_qlora_reference.csv")
print("saved summary:", OUT_DIR / "full_catboost_semantic_summary.json")
print("saved report:", report_path)
